# Complementary Information (charts/maps) for article 1

## Mergging: old feature file with new one - to include "Centroid"(21/06/26)

In [ ]:
import pandas as pd

OLD_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_19_05_26.csv"
NEW_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_conf_0.5_centroid.csv"
OUT_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv"

print("Loading files...")
df_old = pd.read_csv(OLD_PATH)
df_new = pd.read_csv(NEW_PATH)

print(f"Old file : {df_old.shape[0]:,} rows x {df_old.shape[1]:,} columns")
print(f"New file : {df_new.shape[0]:,} rows x {df_new.shape[1]:,} columns")

# --- check image_name overlap ---
images_old = set(df_old["image_name"])
images_new = set(df_new["image_name"])
print(f"\nImages in old only : {len(images_old - images_new):,}")
print(f"Images in new only : {len(images_new - images_old):,}")
print(f"Images in both     : {len(images_old & images_new):,}")

# --- find duplicate columns (excluding image_name) ---
cols_old = set(df_old.columns) - {"image_name"}
cols_new = set(df_new.columns) - {"image_name"}
duplicate_cols = cols_old & cols_new

print(f"\nDuplicate columns found : {len(duplicate_cols)}")
if duplicate_cols:
    print("  Examples:")
    for c in sorted(duplicate_cols)[:10]:
        print(f"    {c}")

# --- merge on image_name, drop duplicate columns from new file ---
df_new_deduped = df_new.drop(columns=list(duplicate_cols))
df_merged = df_old.merge(df_new_deduped, on="image_name", how="inner")

print(f"\nMerged file : {df_merged.shape[0]:,} rows x {df_merged.shape[1]:,} columns")

# --- final check: no duplicate column names ---
dupes_after = df_merged.columns[df_merged.columns.duplicated()].tolist()
print(f"Duplicate columns after merge : {len(dupes_after)}")
if dupes_after:
    print(f"  {dupes_after}")

# --- save ---
df_merged.to_csv(OUT_PATH, index=False)
print(f"\nSaved to : {OUT_PATH}")

## Exploring the merge

In [ ]:
import re
import pandas as pd
import numpy as np

FEATURES_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv"

print("Loading features...")
df = pd.read_csv(FEATURES_PATH)

# clean unnamed columns
unnamed = [c for c in df.columns if str(c).startswith("Unnamed") or not str(c).strip()]
df = df.drop(columns=unnamed) if unnamed else df

all_cols = [c for c in df.columns if c != "image_name"]
df[all_cols] = df[all_cols].fillna(0)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]:,} columns\n")

# ── classify columns ──────────────────────────────────────────────────────────
grid_re = re.compile(r"_\d_\d$")

seg_global = [c for c in all_cols if c.startswith("seg_") and not grid_re.search(c)]
seg_grid   = [c for c in all_cols if c.startswith("seg_") and grid_re.search(c)]
obj_prob   = [c for c in all_cols if c.startswith("obj_") and grid_re.search(c)
              and "_count_" not in c and "_area_" not in c]
obj_count  = [c for c in all_cols if c.startswith("obj_") and "_count_" in c]
obj_area   = [c for c in all_cols if c.startswith("obj_") and "_area_" in c]
stat       = [c for c in all_cols if not c.startswith("seg_") and not c.startswith("obj_")]

groups = {
    "Segmentation global (seg_Class)":            seg_global,
    "Segmentation grid   (seg_Class_r_c)":        seg_grid,
    "Object probability  (obj_Class_r_c)":        obj_prob,
    "Object count        (obj_Class_count_r_c)":  obj_count,
    "Object area         (obj_Class_area_r_c)":   obj_area,
    "Image statistics":                            stat,
}

# ── 1. overview ───────────────────────────────────────────────────────────────
print("=" * 65)
print("1. OVERVIEW")
print("=" * 65)
print(f"\n  Total images   : {df['image_name'].nunique():,}")
print(f"  Total features : {len(all_cols):,}")
print(f"  NaN count      : {df[all_cols].isna().sum().sum():,}\n")
print(f"  {'Group':<48} {'Count':>6}  {'%':>6}")
print("  " + "-" * 63)
for name, cols in groups.items():
    pct = len(cols) / len(all_cols) * 100 if all_cols else 0
    print(f"  {name:<48} {len(cols):>6}  {pct:>5.1f}%")
print("  " + "-" * 63)
print(f"  {'TOTAL':<48} {len(all_cols):>6}")

# ── 2. image statistics ───────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("2. IMAGE STATISTICS")
print("=" * 65)
if not stat:
    print("\n  No image statistics features found.")
else:
    print(f"\n  {len(stat)} features\n")
    print(f"  {'Feature':<35} {'Mean':>10}  {'Std':>10}  {'% non-zero':>10}")
    print("  " + "-" * 70)
    for col in sorted(stat):
        nz   = (df[col] > 0).mean() * 100
        mean = df[col].mean()
        std  = df[col].std()
        print(f"  {col:<35} {mean:>10.4f}  {std:>10.4f}  {nz:>9.1f}%")

# ── 3. segmentation summary ───────────────────────────────────────────────────
print("\n" + "=" * 65)
print("3. SEGMENTATION CLASSES")
print("=" * 65)
seg_classes = sorted([c.replace("seg_", "") for c in seg_global])
print(f"\n  {len(seg_classes)} global classes  |  {len(seg_grid)} grid features\n")
print(f"  {'Class':<35} {'% non-zero':>10}  {'Mean':>10}  {'Grid cols':>10}")
print("  " + "-" * 70)
for cls in seg_classes:
    col       = f"seg_{cls}"
    nz        = (df[col] > 0).mean() * 100
    mean_v    = df[col].mean()
    grid_n    = len([c for c in seg_grid if c.startswith(f"seg_{cls}_")])
    bar       = "█" * int(nz / 5)
    print(f"  {cls:<35} {nz:>9.1f}%  {mean_v:>10.4f}  {grid_n:>10}  {bar}")

# ── 4. object detection summary ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("4. OBJECT DETECTION CLASSES")
print("=" * 65)
obj_classes = sorted(set(
    re.sub(r"_\d_\d$", "", c).replace("obj_", "", 1)
    for c in obj_prob
))
positions = sorted(set(
    re.search(r"(\d_\d)$", c).group(1)
    for c in obj_prob if re.search(r"(\d_\d)$", c)
))
print(f"\n  {len(obj_classes)} classes  |  {len(positions)} grid cells  |  variants: probability, count, area\n")
print(f"  {'Class':<30} {'% any cell':>10}  {'Mean prob':>10}")
print("  " + "-" * 55)
for cls in obj_classes:
    cls_cols = [c for c in obj_prob if re.sub(r"_\d_\d$", "", c) == f"obj_{cls}"]
    nz       = (df[cls_cols] > 0).any(axis=1).mean() * 100 if cls_cols else 0
    mean_p   = df[cls_cols].values.mean() if cls_cols else 0
    if nz > 0:
        bar = "█" * int(nz / 5)
        print(f"  {cls:<30} {nz:>9.1f}%  {mean_p:>10.4f}  {bar}")
    else:
        print(f"  {cls:<30} {nz:>9.1f}%  {mean_p:>10.4f}")

# ── 5. grid cell sparsity ─────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("5. GRID CELL SPARSITY")
print("=" * 65)
rows_ids   = sorted(set(p.split("_")[0] for p in positions))
cols_ids   = sorted(set(p.split("_")[1] for p in positions))
row_labels = {"1": "Top", "2": "Middle", "3": "Bottom"}
col_labels = {"1": "Left", "2": "Centre", "3": "Right"}

for grid_cols, title in [(seg_grid, "Segmentation"), (obj_prob, "Object probability")]:
    print(f"\n  -- {title} (mean % non-zero across classes) --\n")
    print(f"  {'':10} " + "  ".join(f"{col_labels.get(c, c):>10}" for c in cols_ids))
    print("  " + "-" * 43)
    for r in rows_ids:
        vals = []
        for c in cols_ids:
            pos_cols = [col for col in grid_cols if col.endswith(f"_{r}_{c}")]
            v = (df[pos_cols] > 0).mean(axis=0).mean() * 100 if pos_cols else 0
            vals.append(v)
        print(f"  {row_labels.get(r, r):<10} " + "  ".join(f"{v:>9.1f}%" for v in vals))

# ── 6. sparsity by group ──────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("6. SPARSITY SUMMARY BY GROUP")
print("=" * 65)
for name, cols in groups.items():
    if not cols:
        print(f"\n  {name}: no features")
        continue
    nz = (df[cols] > 0).mean(axis=0) * 100
    print(f"\n  {name}")
    print(f"    Features        : {len(cols)}")
    print(f"    Mean % non-zero : {nz.mean():.1f}%")
    print(f"    Min  % non-zero : {nz.min():.1f}%   ({nz.idxmin()})")
    print(f"    Max  % non-zero : {nz.max():.1f}%   ({nz.idxmax()})")

# ── 7. features to drop ───────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("7. FEATURES TO DROP BEFORE MODELLING")
print("=" * 65)
zv      = [c for c in all_cols if df[c].nunique() <= 1]
nzv     = [c for c in all_cols if (df[c] > 0).mean() < 0.001 and c not in zv]
to_drop = set(zv + nzv)
print(f"\n  Zero variance (nunique <= 1)  : {len(zv):,}")
print(f"  Near-zero (< 0.1% non-zero)  : {len(nzv):,}")
print(f"  Total recommended to drop    : {len(to_drop):,}")
print(f"  Remaining after drop         : {len(all_cols) - len(to_drop):,}")

# ── 8. cycling-relevant features ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("8. CYCLING-RELEVANT FEATURES CHECK")
print("=" * 65)
keywords = [
    "Bike Lane", "Bicycle", "Bicyclist", "Sidewalk",
    "Crosswalk - Plain", "Person", "Car", "Road",
    "Parking", "Street Light", "Traffic Sign (Front)",
    "Curb", "Fence", "Vegetation", "Lane Marking - General",
]
print(f"\n  {'Feature':<40} {'Global % nz':>12}  {'Grid cols':>10}  {'In drop list':>12}")
print("  " + "-" * 78)
for kw in keywords:
    global_col = f"seg_{kw}"
    grid_n     = len([c for c in seg_grid if c.startswith(f"seg_{kw}_")])
    if global_col in df.columns:
        nz        = (df[global_col] > 0).mean() * 100
        drop_flag = "YES" if global_col in to_drop else "no"
        print(f"  {global_col:<40} {nz:>11.1f}%  {grid_n:>10}  {drop_flag:>12}")
    else:
        print(f"  seg_{kw:<36} {'MISSING':>12}  {'':>10}  {'':>12}")

print("\nDiagnostic complete.")

# GPT Votes distribution

In [ ]:
"""
GPT-5.2 Vote Breakdown — Descriptive Table
===========================================================
Computes descriptive statistics and raw distribution percentages for GPT-5.2 
votes across specified checkpoints (20k, 40k, 60k, 80k, 100k) without TrueSkill modeling.
"""

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# PATHS
# ============================================================
# Using the 200k combined file so checkpoints above 25k are meaningful 
GPT_100K_PATH_1 = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_100k_20_06_26.csv"
GPT_100K_PATH_2 = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_100k_2_20_06_26.csv"

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
CHECKPOINTS = [20_000, 40_000, 60_000, 80_000, 100_000]

# ============================================================
# HELPERS
# ============================================================
def normalise_image_name(path: str) -> str:
    return os.path.basename(str(path).strip())


def load_votes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.dropna(subset=["vote", "image_name1", "image_name2"]).copy()
    df["image_name1"] = df["image_name1"].apply(normalise_image_name)
    df["image_name2"] = df["image_name2"].apply(normalise_image_name)
    df["vote"]        = df["vote"].astype(int)

    if "voting_time" in df.columns:
        df["voting_time"] = pd.to_datetime(df["voting_time"], format="mixed", dayfirst=True)
        df = df.sort_values("voting_time").reset_index(drop=True)
    return df


# ============================================================
# DATA EXTRACTION AND PROCESSING
# ============================================================
print("Loading GPT-5.2 voting dataset...")
df_gpt_1 = load_votes(GPT_100K_PATH_1)
df_gpt_2 = load_votes(GPT_100K_PATH_2)
df_gpt   = pd.concat([df_gpt_1, df_gpt_2], ignore_index=True)
print(f"  Combined rows available: {len(df_gpt):,}")

table_rows = []

print("\nProcessing checkpoints to compile Descriptive Vote Table...")
for cp in CHECKPOINTS:
    slice_n = min(cp, len(df_gpt))
    if slice_n < cp:
        print(f"  WARNING: checkpoint {cp:,} exceeds available rows, capped at {slice_n:,}")
    df_cp = df_gpt.iloc[:slice_n].copy()

    unique_images = pd.concat([df_cp["image_name1"], df_cp["image_name2"]]).nunique()

    left_count  = (df_cp["vote"] == 1).sum()
    draw_count  = (df_cp["vote"] == 0).sum()
    right_count = (df_cp["vote"] == 2).sum()

    left_pct  = (left_count / slice_n) * 100
    draw_pct  = (draw_count / slice_n) * 100
    right_pct = (right_count / slice_n) * 100

    table_rows.append({
        "Checkpoint Volume": f"{slice_n:,} votes",
        "Unique Images": f"{unique_images:,}",
        "Left Wins (Count)": f"{left_count:,}",
        "Left Wins (%)": f"{left_pct:.2f}%",
        "Draws (Count)": f"{draw_count:,}",
        "Draws (%)": f"{draw_pct:.2f}%",
        "Right Wins (Count)": f"{right_count:,}",
        "Right Wins (%)": f"{right_pct:.2f}%"
    })

df_descriptive_table = pd.DataFrame(table_rows)

print("\n" + "=" * 95)
print("GPT-5.2 VOTE BREAKDOWN — DESCRIPTIVE TABLE")
print("=" * 95)
display(df_descriptive_table)

## SIGMA (variation): Across data sets LLAVA, GPT AND HUMAN

filter application before comparing voting distribution

In [ ]:
# ============================================================
# HUMAN VOTE FILTERING — run this before generating figures
# ============================================================
import numpy as np
import pandas as pd
import os

HUMAN_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data\human_votes_05_06_26.csv"

MIN_VOTES_SPEED    = 5
SPEED_THRESHOLD    = 2.0
MIN_VOTES_PATTERN  = 10
MAX_STREAK         = 10
ALT_RATE_THRESHOLD = 0.90


def normalise_image_name(path):
    return os.path.basename(str(path).strip())


def compute_user_stats(df):
    counts    = df.groupby("user_id").size().rename("vote_count")
    vote_props = (
        df.groupby("user_id")["vote"]
          .value_counts(normalize=True).rename("prop").reset_index()
    )
    max_prop  = vote_props.groupby("user_id")["prop"].max().rename("max_option_prop")
    df_s      = df.sort_values(["user_id", "voting_time"]).copy()
    df_s["dt"] = df_s.groupby("user_id")["voting_time"].diff().dt.total_seconds()
    mean_dt   = df_s.groupby("user_id")["dt"].mean().fillna(np.inf).rename("mean_dt_seconds")
    return pd.concat([counts, max_prop, mean_dt], axis=1)


def compute_pattern_stats(df):
    df_s    = df.sort_values(["user_id", "voting_time"]).copy()
    results = []
    for uid, grp in df_s.groupby("user_id"):
        votes = grp["vote"].tolist()
        n     = len(votes)
        ms = cur = 1
        for i in range(1, n):
            cur = cur + 1 if votes[i] == votes[i-1] else 1
            ms  = max(ms, cur)
        tr = sum(votes[i] != votes[i-1] for i in range(1, n)) / (n-1) if n >= 2 else np.nan
        results.append({"user_id": uid, "max_streak": ms, "transition_rate": tr})
    return pd.DataFrame(results).set_index("user_id")


def compute_bias_threshold(n):
    if   n > 100: return 0.70
    elif n > 50:  return 0.75
    elif n > 10:  return 0.80
    else:         return np.nan


# load
df_raw = pd.read_csv(HUMAN_PATH)
df_raw = df_raw.dropna(subset=["vote", "image_name1", "image_name2", "voting_time"]).copy()
df_raw["image_name1"] = df_raw["image_name1"].apply(normalise_image_name)
df_raw["image_name2"] = df_raw["image_name2"].apply(normalise_image_name)
df_raw["vote"]        = df_raw["vote"].astype(int)
df_raw["voting_time"] = pd.to_datetime(df_raw["voting_time"], format="mixed", dayfirst=True)

# F1 — directional bias
s    = compute_user_stats(df_raw)
s["thr"] = s["vote_count"].apply(compute_bias_threshold)
bad  = s.index[~s["thr"].isna() & (s["max_option_prop"] >= s["thr"])].tolist()
df_f1 = df_raw[~df_raw["user_id"].isin(bad)].copy()

# F2 — speed
s   = compute_user_stats(df_f1)
bad = s.index[(s["vote_count"] >= MIN_VOTES_SPEED) &
              (s["mean_dt_seconds"] < SPEED_THRESHOLD)].tolist()
df_f2 = df_f1[~df_f1["user_id"].isin(bad)].copy()

# F3 — consecutive run
s   = compute_user_stats(df_f2).join(compute_pattern_stats(df_f2))
bad = s.index[(s["vote_count"] >= MIN_VOTES_PATTERN) &
              (s["max_streak"] >= MAX_STREAK)].tolist()
df_f3 = df_f2[~df_f2["user_id"].isin(bad)].copy()

# F4 — alternating pattern
s   = compute_user_stats(df_f3).join(compute_pattern_stats(df_f3))
bad = s.index[(s["vote_count"] >= MIN_VOTES_PATTERN) &
              (s["transition_rate"] >= ALT_RATE_THRESHOLD)].tolist()
df_human_filtered = df_f3[~df_f3["user_id"].isin(bad)].copy()

print(f"Human vote filtering complete.")
print(f"  Before : {len(df_raw):,} votes | {df_raw['user_id'].nunique():,} users")
print(f"  After  : {len(df_human_filtered):,} votes | "
      f"{df_human_filtered['user_id'].nunique():,} users")
print(f"  Removed: {len(df_raw)-len(df_human_filtered):,} votes "
      f"({(len(df_raw)-len(df_human_filtered))/len(df_raw)*100:.1f}%)")
print(f"  Filtered draw rate : {(df_human_filtered['vote']==0).mean()*100:.1f}%")

# Figures for the article

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\paper_exports'
os.makedirs(OUTPUT_DIR, exist_ok=True)

models    = ['Deeper MLP', 'XGBoost', 'LightGBM', 'CatBoost']
gpt_means = [0.6114, 0.6267, 0.6283, 0.6278]
gpt_stds  = [0.0064, 0.0036, 0.0052, 0.0027]
hum_means = [0.4183, 0.4233, 0.4218, 0.4269]
hum_stds  = [0.0084, 0.0105, 0.0061, 0.0022]
baseline  = 1 / 3

BLUE  = '#2171B5'
GREEN = '#1B7837'

x     = np.arange(len(models))
width = 0.32

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars_gpt = ax.bar(
    x - width / 2, gpt_means, width,
    yerr=gpt_stds, capsize=4,
    color=BLUE, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    label='GPT-5.2 (200k votes)'
)

bars_hum = ax.bar(
    x + width / 2, hum_means, width,
    yerr=hum_stds, capsize=4,
    color=GREEN, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    label='Human (19,729 filtered votes)'
)

for bar, mean, std in zip(bars_gpt, gpt_means, gpt_stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        mean + std + 0.007,
        f'{mean*100:.1f}%',
        ha='center', va='bottom',
        fontsize=8.5, fontweight='bold',
        color='black'
    )

for bar, mean, std in zip(bars_hum, hum_means, hum_stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        mean + std + 0.007,
        f'{mean*100:.1f}%',
        ha='center', va='bottom',
        fontsize=8.5, fontweight='bold',
        color='black'
    )

ax.axhline(
    baseline,
    color='#c0392b', linestyle='--',
    linewidth=1.6, alpha=0.9,
    label=f'Random baseline ({baseline*100:.1f}%)'
)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Balanced accuracy (%)', fontsize=11)
ax.set_ylim(0, 0.74)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v*100:.0f}%')
)
ax.set_title(
    'Safety score prediction: balanced accuracy by model and dataset\n'
    '3-class qcut | 1/\u03c3\u00b2 sample weighting | Location-level GroupKFold | 3\u00d75 CV',
    fontsize=11, fontweight='bold', pad=10
)
ax.legend(
    fontsize=10, frameon=True, framealpha=0.95,
    loc='lower right', edgecolor='#cccccc'
)
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.25, linestyle='--', color='#aaaaaa')
ax.tick_params(labelsize=10)

plt.tight_layout()

png_path = os.path.join(OUTPUT_DIR, 'figure_balanced_accuracy_by_model.png')
pdf_path = os.path.join(OUTPUT_DIR, 'figure_balanced_accuracy_by_model.pdf')
fig.savefig(png_path, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
print(f'Saved: {png_path}')
print(f'Saved: {pdf_path}')

plt.show()

# Image year distribution

In [ ]:
import json
import pandas as pd
import os
from collections import Counter

import pandas as pd

COORDS_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\metadata_images_mapping_corrected.csv'

coords_df = pd.read_csv(COORDS_PATH)
coords_df['image_name'] = coords_df['image_blob_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
coords_df['point_id'] = coords_df['image_name'].str.extract(r'^(\d+)_')

from azure.storage.blob import BlobServiceClient

AZURE_CONN_STR = (
    '...'
    '...'
    '...'
)
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client('london')

OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'
os.makedirs(OUTPUT_DIR, exist_ok=True)

unique_blobs = (
    coords_df[['point_id', 'metadata_blob_name']]
    .drop_duplicates(subset='point_id')
    .reset_index(drop=True)
)

results = []
errors  = 0

for i, row in unique_blobs.iterrows():
    pid           = row['point_id']
    metadata_blob = row['metadata_blob_name']
    try:
        data     = container_client.download_blob(metadata_blob).readall()
        meta     = json.loads(data)
        date_str = meta.get('metadata_response', {}).get('date', '')
        if date_str:
            results.append({'point_id': pid, 'year': int(date_str[:4])})
    except Exception:
        errors += 1
    if (i + 1) % 1000 == 0:
        print(f'  Processed {i+1:,} / {len(unique_blobs):,}')

years_df = pd.DataFrame(results)

RIVER_EXCLUDE_IDS = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '4943', '4354', '1610', '3822', '1984'
]
years_df_clean = years_df[~years_df['point_id'].isin(RIVER_EXCLUDE_IDS)].copy()
years_df_clean.to_csv(os.path.join(OUTPUT_DIR, 'image_capture_years.csv'), index=False)

print(f'Locations processed         : {len(unique_blobs):,}')
print(f'Years extracted             : {len(years_df):,}')
print(f'Errors                      : {errors}')
print(f'After excluding river points: {len(years_df_clean):,}')
print(f'Saved to                    : image_capture_years.csv')

In [ ]:
import pandas as pd
import os

FEATURES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv'

feat_df = pd.read_csv(FEATURES_PATH, usecols=['image_name'])
feat_df['image_name'] = feat_df['image_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
feat_df['point_id'] = feat_df['image_name'].str.extract(r'^(\d+)_')

print(f'Total images in features file : {len(feat_df):,}')

# merge each image with its location's year, using the FULL years_df
# (no river exclusion here, this chart describes all captured images)
images_with_year = feat_df.merge(years_df, on='point_id', how='inner')

print(f'Images matched with a year    : {len(images_with_year):,}')
print(f'Images not matched            : {len(feat_df) - len(images_with_year):,}')

In [ ]:
from collections import Counter

year_counts  = Counter(images_with_year['year'].tolist())
sorted_years = sorted(year_counts.keys())
total        = len(images_with_year)

print(f'{"Year":<8}{"Count":>10}{"Share":>10}')
print('-' * 28)
for y in sorted_years:
    n = year_counts[y]
    print(f'{y:<8}{n:>10,}{n/total*100:>9.1f}%')
print('-' * 28)
print(f'{"Total":<8}{total:>10,}{100.0:>9.1f}%')
print(f'\nYear range  : {min(sorted_years)} to {max(sorted_years)}')
print(f'Median year : {int(images_with_year["year"].median())}')
print(f'Most common : {year_counts.most_common(1)[0][0]} '
      f'({year_counts.most_common(1)[0][1]:,} images, '
      f'{year_counts.most_common(1)[0][1]/total*100:.1f}%)')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE        = '#2171B5'
FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

counts = [year_counts[y] for y in sorted_years]
pcts   = [c / total * 100 for c in counts]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars = ax.bar(
    sorted_years, counts,
    color=BLUE, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    width=0.6
)

for bar, val, pct in zip(bars, counts, pcts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts) * 0.008,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=8, fontweight='bold', color='#333333',
        linespacing=1.4
    )

ax2 = ax.twinx()
ax2.set_ylim(0, ax.get_ylim()[1] / total * 100)
ax2.set_ylabel('Share of images (%)', fontsize=10, color='#555555')
ax2.tick_params(labelsize=9, colors='#555555')
ax2.spines[['top', 'right']].set_visible(True)
ax2.spines['right'].set_color('#cccccc')
ax2.spines['top'].set_visible(False)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

ax.set_xlabel('Capture year', fontsize=10)
ax.set_ylabel('Number of images', fontsize=10)
ax.set_title(
    'Google Street View image capture year distribution\n'
    f'Total: {total:,} images across {len(sorted_years)} capture years',
    fontsize=11, fontweight='bold', pad=22
)
ax.set_xticks(sorted_years)
ax.set_xticklabels([str(y) for y in sorted_years], fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax.tick_params(labelsize=9)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'image_capture_year_distribution.pdf')
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

# Image year distribution map 

In [ ]:
import pandas as pd
import os

COORDS_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\metadata_images_mapping_corrected.csv'

coords_df = pd.read_csv(COORDS_PATH)
coords_df['image_name'] = coords_df['image_blob_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
coords_df['point_id'] = coords_df['image_name'].str.extract(r'^(\d+)_')

RIVER_EXCLUDE_IDS = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '4943', '4354', '1610', '3822', '1984'
]

point_coords = (
    coords_df.groupby('point_id')[['actual_lat', 'actual_lng']]
    .first().reset_index()
    .rename(columns={'actual_lat': 'lat', 'actual_lng': 'lng'})
)
point_coords = point_coords[
    ~point_coords['point_id'].isin(RIVER_EXCLUDE_IDS)
].reset_index(drop=True)

print(f'point_coords ready: {len(point_coords):,} locations')

In [ ]:
from collections import Counter

FEATURES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv'

feat_df = pd.read_csv(FEATURES_PATH, usecols=['image_name'])
feat_df['image_name'] = feat_df['image_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
feat_df['point_id'] = feat_df['image_name'].str.extract(r'^(\d+)_').astype(str)

years_df_all = pd.read_csv(
    r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\image_capture_years_all.csv'
)
years_df_all = years_df_all[years_df_all['point_id'] != 'nan'].copy()
years_df_all['point_id'] = years_df_all['point_id'].astype(str)

images_with_year = feat_df.merge(years_df_all, on='point_id', how='inner')
year_counts      = Counter(images_with_year['year'].tolist())
sorted_years     = sorted(year_counts.keys())
total            = len(images_with_year)

print(f'images_with_year : {total:,}')
print(f'years            : {sorted_years}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import geopandas as gpd
import contextily as ctx
import numpy as np
import os, io, requests
from shapely.ops import unary_union

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\paper_exports'
os.makedirs(OUTPUT_DIR, exist_ok=True)

year_map_df = images_with_year.merge(
    point_coords, on='point_id', how='inner'
).drop_duplicates(subset='point_id')

print(f'Locations for map: {len(year_map_df):,}')

gdf_years = gpd.GeoDataFrame(
    year_map_df,
    geometry=gpd.points_from_xy(year_map_df['lng'], year_map_df['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:3857')

london_boundary = None
try:
    url     = ('https://raw.githubusercontent.com/martinjc/UK-GeoJSON/'
               'master/json/administrative/eng/lad.json')
    resp    = requests.get(url, timeout=20)
    gdf_raw = gpd.read_file(io.BytesIO(resp.content))
    london  = gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].copy()
    london_boundary = london.dissolve().to_crs('EPSG:3857')
    boroughs        = london.to_crs('EPSG:3857')
    print('London boundary loaded.')
except Exception as e:
    print(f'Boundary failed: {e}')
    boroughs = None

year_min = int(year_map_df['year'].min())
year_max = int(year_map_df['year'].max())
norm     = mcolors.Normalize(vmin=year_min, vmax=year_max)
cmap     = cm.viridis

fig, (ax_bar, ax_map) = plt.subplots(
    1, 2, figsize=(22, 10),
    gridspec_kw={'wspace': 0.08}
)
fig.patch.set_facecolor('white')
fig.subplots_adjust(top=0.88)

# ── Panel A ───────────────────────────────────────────────────────────────────
ax_bar.set_facecolor('white')

counts_list = [year_counts[y] for y in sorted_years]
pcts_list   = [c / total * 100 for c in counts_list]
bar_colors  = [cmap(norm(y)) for y in sorted_years]
ylim_max    = max(counts_list) * 1.22

bars = ax_bar.bar(
    sorted_years, counts_list,
    color=bar_colors, alpha=0.9,
    edgecolor='white', linewidth=0.5,
    width=0.6
)

for bar, val, pct in zip(bars, counts_list, pcts_list):
    ax_bar.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts_list) * 0.008,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=7, fontweight='bold', color='#333333',
        linespacing=1.4
    )

ax2 = ax_bar.twinx()
ax2.set_ylabel('Share of images (%)', fontsize=10, color='#555555')
ax2.tick_params(labelsize=9, colors='#555555')
ax2.spines[['top']].set_visible(False)
ax2.spines['right'].set_color('#cccccc')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax2.set_ylim(0, ylim_max / total * 100)
ax2.autoscale(False)

ax_bar.set_xlabel('Capture year', fontsize=10)
ax_bar.set_ylabel('Number of images', fontsize=10)
ax_bar.set_xticks(sorted_years)
ax_bar.set_xticklabels([str(y) for y in sorted_years], fontsize=8, rotation=45, ha='right')
ax_bar.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax_bar.spines[['top', 'right']].set_visible(False)
ax_bar.spines['left'].set_color('#cccccc')
ax_bar.spines['bottom'].set_color('#cccccc')
ax_bar.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax_bar.tick_params(labelsize=9)
ax_bar.set_ylim(0, ylim_max)
ax_bar.autoscale(False)
ax_bar.set_title(
    'Panel A — Google Street View image capture year distribution\n'
    f'Total: {total:,} images across {len(sorted_years)} capture years',
    fontsize=11, fontweight='bold', loc='left', pad=10
)

# ── Panel B ───────────────────────────────────────────────────────────────────
ax_map.set_facecolor('white')
ax_map.patch.set_facecolor('white')

xlim = (-57000, 42000)
ylim = (6670000, 6746000)
ax_map.set_xlim(xlim)
ax_map.set_ylim(ylim)

# light grey London fill and borough lines — no basemap for clean white bg
if london_boundary is not None:
    london_boundary.plot(ax=ax_map, color='#f0f0f0',
                         edgecolor='none', zorder=1)
if boroughs is not None:
    boroughs.boundary.plot(ax=ax_map, color='#cccccc',
                           linewidth=0.5, zorder=2)

# scatter points
sc = ax_map.scatter(
    gdf_years.geometry.x,
    gdf_years.geometry.y,
    c=gdf_years['year'],
    cmap=cmap, norm=norm,
    s=14, alpha=0.85,
    linewidths=0, rasterized=True, zorder=5
)

# outer London boundary
if london_boundary is not None:
    london_boundary.boundary.plot(
        ax=ax_map, color='#333333',
        linewidth=1.2, alpha=1.0, zorder=6
    )

ax_map.set_xlim(xlim)
ax_map.set_ylim(ylim)
ax_map.set_axis_off()
ax_map.set_title(
    'Panel B — Spatial distribution of Street View images by capture year',
    fontsize=11, fontweight='bold', loc='left', pad=10
)

# colorbar using exact figure coordinates from diagnostic
cax  = fig.add_axes([0.910, 0.160, 0.012, 0.655])
cbar = fig.colorbar(sc, cax=cax, orientation='vertical')
cbar.set_label('Capture year', fontsize=10)
cbar.set_ticks(range(year_min, year_max + 1, 2))
cbar.ax.tick_params(labelsize=8)

png_path = os.path.join(OUTPUT_DIR, 'figure_image_year_distribution_panels.png')
pdf_path = os.path.join(OUTPUT_DIR, 'figure_image_year_distribution_panels.pdf')
fig.savefig(png_path, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
print(f'Saved: {png_path}')
print(f'Saved: {pdf_path}')

plt.show()

# Figure: Effect of vote scaling 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\paper_exports'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── scaling_df built directly from the confirmed vote-scaling run ─────────────
scaling_df = pd.DataFrame({
    'n_votes':  [20000,  40000,  60000,  80000,  100000, 150000, 200000],
    'MLP_mean': [0.4721, 0.5214, 0.5370, 0.5702, 0.5763, 0.6032, 0.6074],
    'MLP_std':  [0.0090, 0.0142, 0.0145, 0.0155, 0.0151, 0.0073, 0.0095],
    'XGB_mean': [0.4923, 0.5457, 0.5671, 0.5879, 0.6024, 0.6227, 0.6254],
    'XGB_std':  [0.0064, 0.0050, 0.0085, 0.0060, 0.0065, 0.0070, 0.0048],
    'LGB_mean': [0.4886, 0.5381, 0.5655, 0.5819, 0.5981, 0.6224, 0.6215],
    'LGB_std':  [0.0083, 0.0032, 0.0106, 0.0069, 0.0082, 0.0071, 0.0036],
    'CB_mean':  [0.5018, 0.5513, 0.5738, 0.5917, 0.6040, 0.6268, 0.6282],
    'CB_std':   [0.0045, 0.0024, 0.0115, 0.0082, 0.0064, 0.0051, 0.0045],
})

MODEL_COLORS = {
    'Deeper MLP': '#8E44AD',
    'XGBoost':    '#2171B5',
    'LightGBM':   '#E67E22',
    'CatBoost':   '#1B7837',
}

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

x_vals = scaling_df['n_votes'].values / 1000

model_specs = [
    ('Deeper MLP', 'MLP_mean',  'MLP_std',  'o',  MODEL_COLORS['Deeper MLP']),
    ('XGBoost',   'XGB_mean',  'XGB_std',  's',  MODEL_COLORS['XGBoost']),
    ('LightGBM',  'LGB_mean',  'LGB_std',  '^',  MODEL_COLORS['LightGBM']),
    ('CatBoost',  'CB_mean',   'CB_std',   'D',  MODEL_COLORS['CatBoost']),
]

y_all_max = 0
y_all_min = 1

for label, mean_col, std_col, marker, color in model_specs:
    y_means = scaling_df[mean_col].values
    y_stds  = scaling_df[std_col].values
    y_all_max = max(y_all_max, (y_means + y_stds).max())
    y_all_min = min(y_all_min, (y_means - y_stds).min())

    ax.plot(x_vals, y_means,
            color=color, linewidth=2.0,
            marker=marker, markersize=7,
            zorder=5, label=label)
    ax.fill_between(x_vals,
                    y_means - y_stds,
                    y_means + y_stds,
                    alpha=0.10, color=color, zorder=4)

# value labels at final threshold only to avoid clutter
for label, mean_col, std_col, marker, color in model_specs:
    y_means = scaling_df[mean_col].values
    y_stds  = scaling_df[std_col].values
    ax.text(x_vals[-1] + 1.5, y_means[-1],
            f'{y_means[-1]*100:.1f}%',
            ha='left', va='center',
            fontsize=8.5, fontweight='bold', color=color)

ax.set_xlabel('Number of GPT-5.2 synthetic votes (thousands)', fontsize=11)
ax.set_ylabel('Balanced accuracy (%)', fontsize=11)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v*100:.0f}%')
)
ax.set_xticks(x_vals)
ax.set_xticklabels([f'{int(x)}k' for x in x_vals], fontsize=9)
ax.set_xlim(x_vals[0] - 5, x_vals[-1] + 18)
ax.set_ylim(max(0.40, y_all_min - 0.02), y_all_max + 0.05)
ax.set_title(
    'Effect of synthetic vote volume on safety score prediction: all models\n'
    '3-class qcut | 1/\u03c3\u00b2 weighting | 3\u00d75 location-level GroupKFold | '
    'tree hyperparameters tuned on full 200k ranking',
    fontsize=10, fontweight='bold', pad=12, loc='left'
)
ax.legend(fontsize=9, frameon=True, framealpha=0.95,
          loc='lower right', edgecolor='#cccccc')
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax.tick_params(labelsize=9)

plt.tight_layout()

png_path = os.path.join(OUTPUT_DIR, 'figure_vote_scaling_all_models.png')
pdf_path = os.path.join(OUTPUT_DIR, 'figure_vote_scaling_all_models.pdf')
fig.savefig(png_path, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
print(f'Saved: {png_path}')
print(f'Saved: {pdf_path}')

plt.show()